# THE FGD AUTO-ENCODER

This implentation is largly taken from genea_numerical_evaluations repository and written by Youngwoo Yoon, based on the original 
FGD proposed by Youngwoo Yoon, Bok Cha, Joo-Haeng Lee, Minsu Jang, Jaeyeon Lee, Jaehong Kim, and Geehyuk Lee in the 2020 paper: 
"Speech Gesture Generation from the Trimodal Context of Text, Audio, and Speaker Identity"

The link to the original, updated the 20/03/2025 is: 
https://github.com/genea-workshop/genea_numerical_evaluations/blob/2022/FGD

In [ ]:
# Operate from the parent directory
# This allows us to import modules from the parent directory
import os
os.chdir("../../..")

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import clear_output
from collections import defaultdict
from utils.animation.skeleton import Skeleton
from utils.evaluation.FGD.embedding_net import EmbeddingNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else 
    "mps" if torch.backends.mps.is_available() else 
    "cpu")

In [ ]:
def train(
        model_embedding_net_AE: EmbeddingNet,
        training_loader,
        num_epochs: int, 
        lr=0.0005
    ):
    # Add profiling data structures
    profiling = defaultdict(list)
    visalize_step = 512  # How often to print profiling stats

    optimizer = torch.optim.AdamW(model_embedding_net_AE.parameters(), lr=lr)

    torch.set_float32_matmul_precision('high')
    
    # The current lowest validation loss gets defined as an infinitely large number in order to make sure that 
    # it gets reduced in the first epoch. .
    current_min_val_loss = np.inf

    # I then move the model to the device that is being used and put in traning mode
    model_embedding_net_AE = model_embedding_net_AE.to(device)
    # model_embedding_net_AE = torch.compile(model_embedding_net_AE, backend="cudagraphs")
    model_embedding_net_AE.train()

    skeleton: Skeleton = training_loader.dataset.skeleton

    skeleton.set_device(device)
    
    # I then define a map of lists used for tracking the training and validation loss for each epoch. 
    # I'll later use these two sets to plot the progress of the model training.
    loss_rec = {'train' : [], 'val' : [], 'train_plot': []}
    
    # This is the main training loop that goes through the entire dataset and trains the model on it for each epoch.
    for epoch in range(num_epochs):
        progress_bar = tqdm(training_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=True)

        # I reset the training loss for each epoch.
        epoch_train_loss = 0

        training_loader.dataset.reshuffle()  # Shuffle the dataset at the beginning of each epoch

        # During the epoch, all the data items are iterated over.
        for i, batch_data in enumerate(progress_bar):

            original_poses, _, _, _ = [
                item.squeeze(0) for item in batch_data
            ]

            # z-normalize the world space positions
            original_poses = skeleton.normalize_world_positions(original_poses)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                encoded_poses, reconstructed_poses = model_embedding_net_AE(
                    poses = original_poses
                )

            l1_loss = nn.L1Loss(reduction='none')
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                recon_loss = torch.mean(l1_loss(reconstructed_poses, original_poses), dim=(1, 2))  # calc mean over spatial dims

                # Add temporal diff loss - aka punish of the making wrong changes between frames
                target_diff = original_poses[:, 1:] - original_poses[:, :-1]
                recon_diff = reconstructed_poses[:, 1:] - reconstructed_poses[:, :-1]
                recon_loss += torch.mean(l1_loss(recon_diff, target_diff), dim=(1, 2))  

                recon_loss = torch.sum(recon_loss)  # Sum over all elements

            epoch_train_loss += recon_loss.item()
            progress_bar.set_postfix({'loss': recon_loss.item()})
            loss_rec['train'].append(recon_loss.item())

            if i % visalize_step == 0:
                
                clear_output(wait=True)

                # add the averaged loss over hte last visalize_step to the loss_rec['train_plot']
                loss_rec['train_plot'].append(np.mean(loss_rec['train'][-visalize_step:]))
                
                # Visualization code remains unchanged
                fig, axs = plt.subplots(1, 5, figsize=(30, 6))

                cmap = 'viridis'
                vmin = -3
                vmax = 3

                axs[0].imshow(reconstructed_poses.to(torch.float32).permute(0, 2, 1)[0, :, :].cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
                axs[0].set_title("Output tensor")
                axs[0].text(10, 190, f"Max: {torch.max(reconstructed_poses):.4f}", color="black")
                axs[0].text(10, 200, f"Min: {torch.min(reconstructed_poses):.4f}", color="black")
                axs[0].text(10, 210, f"Mean: {torch.mean(reconstructed_poses):.4f}", color="black")

                axs[1].imshow(original_poses.to(torch.float32).permute(0, 2, 1)[0, :, :].cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
                axs[1].set_title("Actual gesture")
                axs[1].text(10, 190, f"Max: {torch.max(original_poses):.4f}", color="black")
                axs[1].text(10, 200, f"Min: {torch.min(original_poses):.4f}", color="black")
                axs[1].text(10, 210, f"Mean: {torch.mean(original_poses):.4f}", color="black")

                axs[2].imshow((original_poses - reconstructed_poses).to(torch.float32).permute(0, 2, 1)[0, :, :].cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
                axs[2].set_title("Difference between output and actual gesture")
                axs[2].text(10, 190, f"Max: {torch.max(original_poses - reconstructed_poses):.4f}", color="black")

                axs[3].plot(loss_rec['train_plot'])
                axs[3].set_title('Training Loss')
                axs[3].set_xlabel('Epoch')
                axs[3].set_ylabel('Loss')
                axs[3].set_yscale('log')
                axs[3].grid(True)
                
                plt.show()

                # Save the model
                torch.save(model_embedding_net_AE.state_dict(), f"fgd_model_30.pth")

            optimizer.zero_grad()
            recon_loss.backward()

            optimizer.step()
    
    # When all of the epochs are over, the entire list of training loss and validation loss are returned.
    return loss_rec['train'] #, loss_rec['val']

### Using the traing loop and model

In [ ]:
from dataset.dataset import *
# Number of bones = 58
# Number of channels per bone (for world position) = 3
# posedim = 58 * 3 = 174

model_AE = EmbeddingNet(pose_dim=174, n_frames=30)  # pose_features_per_frame and n_gesture_length of our model

train_loss = train(
    model_embedding_net_AE              = model_AE,
    training_loader                     = DataLoader(
                                            GPUDataset(
                                                consolidated_file= "dataset/genea2023_dataset/trn/main-agent/consolidated.npz", # or "dataset/genea2023_dataset/trn/main-agent/training_windows_100k.npz"
                                                seq_length=30,
                                                seed_length=0,
                                                batch_size=256,
                                                epoch_length=512,
                                                use_world_pos_gesture_features=True,
                                                device=device
                                            ),
                                            batch_size = 1,     # Keep at 1 since dataset handles batching
                                            num_workers = 0,    # Must be 0 for GPU tensors
                                            pin_memory = False  # Not needed for GPU data
                                        ),
    num_epochs                          = 20,
    lr                                  = 0.003
)

In [ ]:
# Save the model
model_AE_path = "utils/evaluation/FGD/models/fgd_model_30.pth"
torch.save(model_AE.state_dict(), model_AE_path)

## Visualize Global World Position Statistics and Outliers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
import torch
from DiffuseStyleGestureReproduced.utils.animation.skeleton import Skeleton

# Load the global world position data from the consolidated file
consolidated_file = "dataset/genea2023_dataset/trn/main-agent/consolidated.npz"
data = np.load(consolidated_file)
gestures = data["gestures"]

# Load metadata
meta_file = consolidated_file.replace('.npz', '_meta.pkl')
with open(meta_file, 'rb') as f:
    metadata = pickle.load(f)
    skeleton: Skeleton = metadata['skeleton']
    mean_pose = skeleton.mean_pose.numpy()
    std_pose = skeleton.std_pose.numpy()


# calculate world positions

print("Gestures shape:", gestures.shape)
print("Gestures dtype:", gestures.dtype)
world_pos_gestures = skeleton.calculate_world_positions(gestures).numpy()

# print skeleton data
print("Skeleton data - joint offsets", skeleton.joint_offsets)
print("Skeleton data - joint position indices", skeleton.joint_position_indices)
print("Skeleton data - joint rotation indices", skeleton.joint_rotation_indices)
print("Skeleton data - joints", skeleton.joints)
print("Skeleton data - joint parents", skeleton.joint_parent)
print("Skeleton data - joint channels", skeleton.joint_channels)
print("Skeleton data - joint categories", skeleton.bone_categories)
print("Skeleton data - bone indices map", skeleton.bone_to_indices_map)
print("Skeleton data - target joints", skeleton.target_joints)
print("Skeleton data - end sites", skeleton.end_sites)
print("Skeleton data - mean pose", skeleton.mean_pose)
print("Skeleton data - std pose", skeleton.std_pose)
print("Skeleton data - world pos mean", skeleton.world_pos_mean_pose)
print("Skeleton data - world pos std", skeleton.world_pos_std_pose)
print("Skeleton data - device", skeleton.device)

# print the first 20 gesture values
print("First 20 gesture values:")
print(gestures[:20])

# print the first 20 world positions
print("First 20 world positions:")
print(world_pos_gestures[:20])

# Print share of values in the range of -1 to 1, -2 to 2, -3 to 3
print("Fraction in range -1 to 1:", np.sum((world_pos_gestures > -1) & (world_pos_gestures < 1)) / world_pos_gestures.size)
print("Fraction in range -2 to 2:", np.sum((world_pos_gestures > -2) & (world_pos_gestures < 2)) / world_pos_gestures.size)
print("Fraction in range -3 to 3:", np.sum((world_pos_gestures > -3) & (world_pos_gestures < 3)) / world_pos_gestures.size)

# z-normalize the world positions
world_pos_gestures = skeleton.normalize_world_positions(torch.tensor(world_pos_gestures)).numpy()

print("After z-normalization:")

# Print share of values in the range of -1 to 1, -2 to 2, -3 to 3
print("Fraction in range -1 to 1:", np.sum((world_pos_gestures > -1) & (world_pos_gestures < 1)) / world_pos_gestures.size)
print("Fraction in range -2 to 2:", np.sum((world_pos_gestures > -2) & (world_pos_gestures < 2)) / world_pos_gestures.size)
print("Fraction in range -3 to 3:", np.sum((world_pos_gestures > -3) & (world_pos_gestures < 3)) / world_pos_gestures.size)

# print if skeleton.std_pose contains NaNs
print("Any NaNs in skeleton.std_pose?", np.any(np.isnan(skeleton.std_pose.numpy())))
# print if skeleton.std_pose contains infs
print("Any infs in skeleton.std_pose?", np.any(np.isinf(skeleton.std_pose.numpy())))
# print if skeleton.std_pose contains -inf
print("Any -infs in skeleton.std_pose?", np.any(skeleton.std_pose.numpy() == -np.inf))
# print if skeleton.std_pose contains 0s
print("Any 0s in skeleton.std_pose?", np.any(skeleton.std_pose.numpy() == 0))

# Print if world_pos_gestures contains NaNs
print("Any NaNs in world_pos_gestures?", np.any(np.isnan(world_pos_gestures)))

# print if world_pos_gestures contains infs
print("Any infs in world_pos_gestures?", np.any(np.isinf(world_pos_gestures)))
# print if world_pos_gestures contains -inf
print("Any -infs in world_pos_gestures?", np.any(world_pos_gestures == -np.inf))

# Print share of values in the range of -1 to 1, -2 to 2, -3 to 3
print("Fraction in range -1 to 1:", np.sum((world_pos_gestures > -1) & (world_pos_gestures < 1)) / world_pos_gestures.size)
print("Fraction in range -2 to 2:", np.sum((world_pos_gestures > -2) & (world_pos_gestures < 2)) / world_pos_gestures.size)
print("Fraction in range -3 to 3:", np.sum((world_pos_gestures > -3) & (world_pos_gestures < 3)) / world_pos_gestures.size)

# world_pos_gestures = (world_pos_gestures - mean_pose) / std_pose

# If you have a separate world position array, use that instead:
# world_pos_gestures = data["bvh_worldpos_features"]

# Plot histograms of the mean and std across all dimensions
means = np.mean(world_pos_gestures.astype(np.float64), axis=0, dtype=np.float64)
stds = np.std(world_pos_gestures.astype(np.float64), axis=0, dtype=np.float64)

print("Calculated means (first 20 dims):", means[:20])
print("Calculated stds (first 20 dims):", stds[:20])

print("Stored means (first 20 dims):", skeleton.world_pos_mean_pose[:20])
print("Stored stds (first 20 dims):", skeleton.world_pos_std_pose[:20])

# print skeleton data
print("Skeleton data - joint offsets", skeleton.joint_offsets)


plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(means, bins=50)
plt.title("Histogram of Means (all dims)")
plt.xlabel("Mean")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
plt.hist(stds, bins=50)
plt.title("Histogram of Stds (all dims)")
plt.xlabel("Std")
plt.ylabel("Count")
plt.show()

# Plot value distributions for all dimensions
dims_to_plot = range(world_pos_gestures.shape[1])  # All dimensions
plt.figure(figsize=(15, 160))
for i, dim in enumerate(dims_to_plot):
    plt.subplot(int(np.ceil(world_pos_gestures.shape[1] / 5)), 5, i+1)
    plt.hist(world_pos_gestures[:, dim], bins=100)
    plt.title(f"Dim {dim}")
    plt.xlabel("Value")
    plt.ylabel("Count")
plt.tight_layout()
plt.show()